In [ ]:
import random
import json
from datetime import datetime
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
import pycountry

from emu_renewal.constants import DATA_PATH, FULL_RUN, TIMEOUTS, RERUNS, OUTPUTS_PATH, ANALYSIS_NAMES
from emu_renewal.outputs import get_param_vals_by_analysis
from emu_renewal.utils import get_analysis_paths

In [ ]:
countries = json.load(open(DATA_PATH / f"config/included.json", "r"))
analysis_paths = get_analysis_paths(RERUNS + FULL_RUN + TIMEOUTS, countries)
n_samples = 20

In [ ]:
waning_paths = {k: v for k, v in get_analysis_paths(["50165821"], countries).items() if v}
waning_sample_countries = random.sample(list(waning_paths), n_samples)
sample_paths = {c: {} for c in waning_sample_countries}
for iso3 in waning_sample_countries:
    sample_mob_type = random.sample(list(waning_paths[iso3]), 1)[0]
    sample_paths[iso3][sample_mob_type] = waning_paths[iso3][sample_mob_type]

In [ ]:
sample_paths[iso3]

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(20, 15))
flat_axes = axes.ravel()
for c, iso3 in enumerate(sample_paths):
    sample_path = sample_paths[iso3]
    analysis_path = analysis_paths[iso3]
    mob_type = next(iter(sample_path))
    run_paths = {"waning": sample_path, "no_waning": analysis_path}
    procs = {p: pd.read_hdf(run_paths[p][mob_type] / "spaghetti.h5")["process"] for p in run_paths}
    quants = {p: procs[p].quantile([0.5], axis=1).T for p in run_paths}
    
    ax = flat_axes[c]
    for p in run_paths:
        ax.plot(quants[p].index, quants[p], label=p)
    ax.legend()
    ax.set_title(f"{pycountry.countries.lookup(iso3).name},  {ANALYSIS_NAMES[mob_type]}")
    ax.set_yticks([])
    ax.set_ylabel("")
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=70)
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(20, 15))
flat_axes = axes.ravel()
param = "dispersion_proc"
for c, iso3 in enumerate(sample_paths):
    sample_path = sample_paths[iso3]
    analysis_path = analysis_paths[iso3]
    mob_type = next(iter(sample_path))
    run_paths = {"waning": sample_path, "no_waning": analysis_path}
    posts = [get_param_vals_by_analysis(param, p)[mob_type] for p in run_paths.values()]
    combined_disps = pd.concat(posts, axis=1)
    combined_disps.columns = run_paths.keys()
    
    ax = flat_axes[c]
    sns.kdeplot(combined_disps, ax=ax, fill=True, alpha=0.1, linewidth=1.5, common_norm=False)
    ax.set_title(f"{pycountry.countries.lookup(iso3).name},  {ANALYSIS_NAMES[mob_type]}")
    ax.set_yticks([])
    ax.set_ylabel("")
fig.tight_layout()